# Feature Selection Analysis -- Telco Churn
## Module 3, Class 4

**Objective:** Compare three feature selection methods and measure their effect on model performance.

### What you will practice
- Correlation filtering (drop redundant features)
- Mutual Information scoring
- Recursive Feature Elimination (RFE)
- Comparing models trained on different feature subsets

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Setup complete.")

## 1. Load Data with Engineered Features

We reproduce the same cleaning and feature engineering from Class 3.

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Clean
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop('customerID', axis=1)

# Engineer features (same as Class 3)
df['charge_per_month'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
df['monthly_charge_x_tenure'] = df['MonthlyCharges'] * df['tenure']
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['num_services'] = df[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)
df['tenure_years'] = df['tenure'] / 12
df['charge_per_service'] = df['MonthlyCharges'] / df['num_services'].replace(0, 1)

# One-hot encode
cat_cols = df.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Total features: {X.shape[1]}")
print(f"Features: {list(X.columns)}")

In [ ]:
# Train/test split (consistent across all experiments)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

---
## 2. Method 1: Correlation Filtering

If two features are highly correlated (>0.85), they carry nearly the same information. We can drop one without losing much signal.

In [ ]:
# Compute correlation matrix on training data
corr_matrix = X_train.corr().abs()

# Find pairs with correlation > 0.85
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper.columns:
    for idx in upper.index:
        val = upper.loc[idx, col]
        if val > 0.85:
            high_corr_pairs.append((idx, col, val))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
high_corr_df = high_corr_df.sort_values('Correlation', ascending=False)
print(f"Highly correlated pairs (>0.85):")
high_corr_df

In [ ]:
# Drop one feature from each highly correlated pair
# Strategy: drop the one that appears more frequently in high-corr pairs
to_drop = set()
for _, row in high_corr_df.iterrows():
    if row['Feature 1'] not in to_drop and row['Feature 2'] not in to_drop:
        # Drop the second one (arbitrary but consistent)
        to_drop.add(row['Feature 2'])

print(f"Dropping {len(to_drop)} features: {to_drop}")

corr_features = [c for c in X.columns if c not in to_drop]
print(f"Remaining features: {len(corr_features)}")

---
## 3. Method 2: Mutual Information

Mutual Information measures how much knowing a feature reduces uncertainty about the target. Unlike correlation, it captures **non-linear** relationships too.

In [ ]:
# Compute MI scores
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({'Feature': X.columns, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=False)

print("Top 15 features by Mutual Information:")
mi_df.head(15)

In [ ]:
# Plot top 15
top15 = mi_df.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15['Feature'][::-1], top15['MI Score'][::-1], color='steelblue', edgecolor='black')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 15 Features by Mutual Information')
plt.tight_layout()
plt.show()

In [ ]:
# Select top 15 MI features
mi_features = mi_df.head(15)['Feature'].tolist()
print(f"MI-selected features ({len(mi_features)}): {mi_features}")

---
## 4. Method 3: Recursive Feature Elimination (RFE)

RFE trains a model, removes the least important feature, repeats until the desired number of features remains.

In [ ]:
# RFE with RandomForest (select 15 features to match MI)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=15, step=1)
rfe.fit(X_train, y_train)

rfe_mask = rfe.support_
rfe_features = X.columns[rfe_mask].tolist()
print(f"RFE-selected features ({len(rfe_features)}):")
for f in rfe_features:
    print(f"  - {f}")

In [ ]:
# Feature ranking (1 = selected, higher = eliminated earlier)
rfe_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')
rfe_ranking.head(20)

---
## 5. Model Comparison

Train LogisticRegression on 4 feature sets:
1. All features
2. Correlation-filtered features
3. MI-selected top 15
4. RFE-selected top 15

In [ ]:
def train_and_evaluate(X_tr, X_te, y_tr, y_te, label):
    """Scale, train LogisticRegression, return F1 for churn class."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_te_s)
    
    f1 = f1_score(y_te, y_pred)
    print(f"{label}: F1={f1:.4f} | {X_tr.shape[1]} features")
    return f1

results = {}

# All features
results['All Features'] = train_and_evaluate(
    X_train, X_test, y_train, y_test, 'All Features')

# Correlation-filtered
results['Corr Filtered'] = train_and_evaluate(
    X_train[corr_features], X_test[corr_features], y_train, y_test, 'Corr Filtered')

# MI-selected
results['MI Top 15'] = train_and_evaluate(
    X_train[mi_features], X_test[mi_features], y_train, y_test, 'MI Top 15')

# RFE-selected
results['RFE Top 15'] = train_and_evaluate(
    X_train[rfe_features], X_test[rfe_features], y_train, y_test, 'RFE Top 15')

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Method': list(results.keys()),
    'Num Features': [
        X_train.shape[1],
        len(corr_features),
        len(mi_features),
        len(rfe_features)
    ],
    'F1 Score': list(results.values())
})
comparison['F1 Score'] = comparison['F1 Score'].round(4)
comparison

In [ ]:
# Which features are selected by BOTH MI and RFE?
overlap = set(mi_features) & set(rfe_features)
print(f"Features selected by both MI and RFE ({len(overlap)}):")
for f in sorted(overlap):
    print(f"  - {f}")

print(f"\nOnly in MI: {set(mi_features) - set(rfe_features)}")
print(f"Only in RFE: {set(rfe_features) - set(mi_features)}")

---
## TODO: Discussion

Answer these questions:

1. Which features appeared in both MI and RFE selections? Why do you think these are consistently important for predicting churn?
2. Did reducing features hurt performance? By how much?
3. When would you prefer MI over RFE, and vice versa?
4. If you had to pick just 5 features for a production model, which would you choose and why?

*TODO: Write your answers here.*

1. ...
2. ...
3. ...
4. ...

---
## Summary

| Method | How it works | Pros | Cons |
|--------|-------------|------|------|
| Correlation filtering | Drops redundant features (>0.85 corr) | Fast, simple | Only catches linear redundancy |
| Mutual Information | Scores each feature's info about target | Captures non-linear relationships | Doesn't account for feature interactions |
| RFE | Iteratively removes least important feature | Model-aware selection | Slow on large datasets |